# 안전귀가Navi — Day 2 전처리 노트북

**입력:** `data/raw/` 5종 공공데이터  
**출력:** `data/processed/` 통합 데이터 3개 + 검증 결과

## 처리 흐름
1. raw 5종 로드 + 통합 스키마 변환
2. 좌표 청소 (서울 범위 외 39건 제거)
3. 안심귀갓길 시설물 코드(301~308) 분해 → CCTV/조명/긴급/정책 보강
4. KDTree 인덱스 빌드 (EPSG:5179 미터 좌표)
5. parquet + pickle 저장
6. 샘플 좌표(강남역) 검색 검증

In [1]:
import sys
from pathlib import Path

# src 모듈 임포트
sys.path.insert(0, str(Path(r'C:\Users\pjhic\Projects\Seoul_bigdata\src')))
import preprocess

# Windows 한글 출력 안정화
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import pandas as pd
import geopandas as gpd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(f'RAW : {preprocess.RAW}')
print(f'OUT : {preprocess.PROCESSED}')
print(f'시설코드 매핑 ({len(preprocess.FACILITY_CODE_MAP)}종):')
for code, (t, st) in preprocess.FACILITY_CODE_MAP.items():
    print(f'  {code} → type={t:8s}  sub_type={st}')

RAW : C:\Users\pjhic\Projects\Seoul_bigdata\data\raw
OUT : C:\Users\pjhic\Projects\Seoul_bigdata\data\processed
시설코드 매핑 (8종):
  301 → type=bell      sub_type=안심귀갓길_안심벨
  302 → type=cctv      sub_type=안심귀갓길_CCTV
  303 → type=facility  sub_type=안내표지판
  304 → type=facility  sub_type=노면표기
  305 → type=facility  sub_type=안심귀갓길_서비스안내판
  306 → type=facility  sub_type=112_위치신고_안내판
  307 → type=light     sub_type=안심귀갓길_보안등
  308 → type=facility  sub_type=기타


## Step 1. raw 5종 로드 + 변환
각 데이터를 통합 스키마(`type, sub_type, lat, lon, gu, dong, source, extra`)로 변환.

In [2]:
cctv_unified = preprocess.cctv_to_unified(preprocess.load_raw_cctv())
print(f'CCTV       : {len(cctv_unified):>7,} 행')

light_unified = preprocess.streetlight_to_unified(preprocess.load_raw_streetlight())
print(f'가로등     : {len(light_unified):>7,} 행')

bell_unified = preprocess.emergency_bell_to_unified(preprocess.load_raw_emergency_bell())
print(f'안전비상벨 : {len(bell_unified):>7,} 행')

fac_gdf = preprocess.load_safe_route_facilities()
fac_unified = preprocess.facilities_to_unified(fac_gdf)
print(f'안심귀갓길 시설물: {len(fac_unified):>3,} 행 (시설코드별 분배 후)')

CCTV       :  48,751 행


가로등     :  19,291 행


안전비상벨 :  20,745 행


안심귀갓길 시설물: 11,883 행 (시설코드별 분배 후)


In [3]:
# 통합 — 안심귀갓길 시설물 분배 결과 확인
print('안심귀갓길 시설물 분배 (type 기준):')
print(fac_unified['type'].value_counts())
print('\n안심귀갓길 시설물 sub_type 상위:')
print(fac_unified['sub_type'].value_counts())

안심귀갓길 시설물 분배 (type 기준):
type
facility    7637
light       1758
cctv        1487
bell        1001
Name: count, dtype: int64

안심귀갓길 시설물 sub_type 상위:
sub_type
안심귀갓길_서비스안내판    5618
안심귀갓길_보안등       1758
노면표기            1498
안심귀갓길_CCTV      1487
안심귀갓길_안심벨       1001
안내표지판            376
기타                75
112_위치신고_안내판      70
Name: count, dtype: int64


## Step 2. 통합 점 데이터셋 빌드

In [4]:
all_facilities = pd.concat([cctv_unified, light_unified, bell_unified, fac_unified],
                            ignore_index=True)[preprocess.UNIFIED_COLUMNS]
print(f'통합 점 데이터: {len(all_facilities):,} 행, {len(all_facilities.columns)} 컬럼\n')
print('type별 분포:')
print(all_facilities['type'].value_counts())
print('\nsource별 분포:')
print(all_facilities['source'].value_counts())

통합 점 데이터: 100,670 행, 8 컬럼

type별 분포:
type
cctv        50238
bell        21746
light       21049
facility     7637
Name: count, dtype: int64

source별 분포:
source
cctv_national          48751
emergency_bell         20745
streetlight_seoul      19291
safe_route_facility    11883
Name: count, dtype: int64


In [5]:
# 자치구 커버리지 (가로등은 자치구 정보 없음 → 제외)
with_gu = all_facilities[all_facilities['gu'].notna()]
print(f'자치구 커버리지: {with_gu["gu"].nunique()}/25개')
print('\n자치구별 type 분포 (Top 5):')
pivot = with_gu.groupby(['gu', 'type']).size().unstack(fill_value=0)
pivot['합계'] = pivot.sum(axis=1)
print(pivot.sort_values('합계', ascending=False))

자치구 커버리지: 25/25개

자치구별 type 분포 (Top 5):
type  bell  cctv  facility  light    합계
gu                                     
영등포구  1348  5358       372     75  7153
양천구    988  4816       129     42  5975
강북구     99  4452       607    177  5335
성북구   1484  2422       463    107  4476
도봉구   1077  2839       219     44  4179
용산구   1456  1996       412     94  3958
서초구     99  3313       420    125  3957
강남구   1351  1915       526    111  3903
구로구   1458  2171       188     58  3875
노원구    109  3171       313    109  3702
중랑구   1398  1850       295     58  3601
관악구   1049  1688       381     98  3216
강서구    964  1848       240     57  3109
마포구   1294  1526       171     48  3039
광진구   1149  1337       177     30  2693
동대문구   872  1367       321     36  2596
서대문구   745  1362       285     73  2465
중구     764  1194       234     62  2254
동작구    685   926       398     75  2084
금천구    731   962       140     21  1854
송파구     48  1255       401     58  1762
종로구    100  1097       298     64  1559


In [6]:
# 강동구 CCTV 보강 확인 — 안심귀갓길 시설물에서 강동구 + type=cctv
gangdong_cctv = all_facilities[(all_facilities['gu'] == '강동구') & (all_facilities['type'] == 'cctv')]
print(f'강동구 CCTV (안심귀갓길 시설물 보강): {len(gangdong_cctv)} 건')
print(f'성동구 CCTV (안심귀갓길 시설물 보강): {len(all_facilities[(all_facilities["gu"]=="성동구") & (all_facilities["type"]=="cctv")])} 건')
print('→ 누락 자치구도 0건은 면함')

강동구 CCTV (안심귀갓길 시설물 보강): 48 건


성동구 CCTV (안심귀갓길 시설물 보강): 36 건
→ 누락 자치구도 0건은 면함


## Step 3. 안심귀갓길 경로 정리

In [7]:
routes_gdf = preprocess.load_safe_routes()
routes = preprocess.routes_to_unified(routes_gdf)
print(f'안심귀갓길 경로: {len(routes)} 행, CRS: {routes.crs}')
print(f'컬럼: {list(routes.columns)}\n')
routes.drop(columns='geometry').head()

안심귀갓길 경로: 362 행, CRS: EPSG:4326
컬럼: ['route_id', 'route_name', 'gu', 'dong', 'length_m', 'bell_cnt', 'cctv_cnt', 'light_cnt', 'sign_cnt', 'create_year', 'detail_loc', 'geometry']



,route_id,route_name,gu,dong,length_m,bell_cnt,cctv_cnt,light_cnt,sign_cnt,create_year,detail_loc
0,1111011000_04,종로안심04,종로구,누하동,305.00,4,13,14,0,2015,필운대로5길
1,1111011200_03,종로안심03,종로구,체부동,211.25,1,8,7,0,2015,자하문로5길
2,1111014900_01,종로안심01,종로구,원서동,481.92,3,5,26,0,2017,창덕궁1가길
3,1111016800_04,혜화안심04,종로구,동숭동,133.55,2,2,7,0,2017,동숭4나길
4,1111016900_01,혜화안심01,종로구,혜화동,247.78,2,2,15,0,2015,혜화로 6길


## Step 4. KDTree 인덱스 빌드
좌표를 EPSG:5179(미터)로 변환 후 KDTree 구축 → 미터 단위 반경 검색.

In [8]:
kdtree_data = preprocess.build_kdtree_index(all_facilities)
print(f'KDTree 빌드 완료: {len(kdtree_data["coords_5179"]):,} points (CRS={kdtree_data["crs_meters"]})')

KDTree 빌드 완료: 100,670 points (CRS=EPSG:5179)


## Step 5. 저장

In [9]:
preprocess.save_all(all_facilities, routes, kdtree_data)

for f in sorted(preprocess.PROCESSED.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<35} {size_kb:>10,.1f} KB')

  all_facilities.parquet                 1,526.0 KB
  kdtree_index.pkl                       5,091.0 KB
  safe_routes.parquet                       99.6 KB


## Step 6. 검증 — 샘플 좌표 반경 검색
강남역(37.4979, 127.0276)과 신촌역(37.5559, 126.9367)에서 반경 300m 검색이 정상 동작하는지 확인.

In [10]:
import time

# 처음부터 다시 로드 (실제 점수 엔진처럼)
facilities, routes_loaded, idx = preprocess.load_processed()
tree = idx['kdtree']

samples = {
    '강남역':  (37.4979, 127.0276),
    '신촌역':  (37.5559, 126.9367),
    '잠실역':  (37.5133, 127.1000),
}

for name, (lat, lon) in samples.items():
    x, y = preprocess.transform_query_point(lat, lon)
    t0 = time.perf_counter()
    nearby_idx = tree.query_ball_point((x, y), r=300)
    elapsed = (time.perf_counter() - t0) * 1000

    counts = facilities.iloc[nearby_idx]['type'].value_counts().to_dict()
    print(f'{name} 반경 300m  ({elapsed:.2f} ms)  총 {len(nearby_idx)}건')
    for t in ['cctv', 'light', 'bell', 'facility']:
        print(f'    {t:8s}: {counts.get(t, 0):>4}')

강남역 반경 300m  (0.19 ms)  총 57건
    cctv    :   49
    light   :    0
    bell    :    8
    facility:    0
신촌역 반경 300m  (0.09 ms)  총 101건
    cctv    :   46
    light   :    6
    bell    :   23
    facility:   26
잠실역 반경 300m  (0.08 ms)  총 3건
    cctv    :    3
    light   :    0
    bell    :    0
    facility:    0


In [11]:
# 안심귀갓길 거리 계산 검증 — 강남역에서 가장 가까운 경로
from shapely.geometry import Point
import geopandas as gpd

gangnam_pt = gpd.GeoSeries([Point(127.0276, 37.4979)], crs='EPSG:4326').to_crs(epsg=5179)
routes_5179 = routes_loaded.to_crs(epsg=5179)
distances = routes_5179.geometry.distance(gangnam_pt.iloc[0])

nearest_idx = distances.idxmin()
print(f'강남역에서 가장 가까운 안심귀갓길:')
print(f'  {routes_loaded.iloc[nearest_idx]["route_name"]} ({routes_loaded.iloc[nearest_idx]["gu"]})')
print(f'  거리: {distances.min():.1f}m')
print(f'  300m 내 안심귀갓길: {(distances < 300).sum()}개')

강남역에서 가장 가까운 안심귀갓길:
  강남안심01 (강남구)
  거리: 558.6m
  300m 내 안심귀갓길: 0개


## Step 7. 최종 요약

In [12]:
summary = pd.DataFrame([
    {'산출물': 'all_facilities.parquet', '행수': len(all_facilities), '내용': '통합 점 데이터 (5종 결합)'},
    {'산출물': 'safe_routes.parquet',     '행수': len(routes),         '내용': '안심귀갓길 경로 (LineString)'},
    {'산출물': 'kdtree_index.pkl',        '행수': len(kdtree_data['coords_5179']), '내용': 'KDTree (EPSG:5179)'},
])
summary

,산출물,행수,내용
0,all_facilities.parquet,100670,통합 점 데이터 (5종 결합)
1,safe_routes.parquet,362,안심귀갓길 경로 (LineString)
2,kdtree_index.pkl,100670,KDTree (EPSG:5179)
